# 02 – Topic Modeling

**Goal:** discover what people discuss in the tech and politics posts, without giving the model any labels in advance (unsupervised learning).

**Input:** `data/posts_clean.csv`
**Output:** `data/posts_with_topics.csv`, `data/topic_info.csv`

**Method: BERTopic**
1. Convert each post to a numeric vector (embedding) that captures its meaning
2. Reduce the dimensions of those vectors (UMAP)
3. Cluster nearby vectors (HDBSCAN), where each cluster is a topic
4. Describe each topic with its most distinctive words (c-TF-IDF)

**Evaluation:** topic coherence (c_v), topic diversity, and manual reading of the topics.

## 1. Load the clean posts

In [62]:
import pandas as pd
import numpy as np

posts = pd.read_csv("data/posts_clean.csv")
posts = posts.reset_index(drop=True)
docs = posts["text_clean"].tolist()

print("posts:", len(docs))
posts[["text_clean", "seed_domain"]].head()

posts: 20452


,text_clean,seed_domain
0,Artificial intelligence is now cloning videos ...,tech
1,"Also, the way they distinguish between the *tr...",tech
2,Did you know artificial intelligence can now c...,tech
3,Major US Banks Begin Blocking Third-Party AI A...,tech
4,New reports show artificial intelligence can n...,tech


## 2. Turn posts into embeddings

A computer can't compare sentences directly, so we convert each post into a list of 384 numbers using the pretrained model `all-MiniLM-L6-v2`. Posts with similar meaning get similar numbers, even when they use different words ("AI regulation" and "rules for artificial intelligence" end up close together).

We use an English model because we collected English posts only. We save the embeddings to disk so we never need to recompute them (this is the slowest step).

In [63]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(docs, batch_size=64, show_progress_bar=True)

np.save("data/embeddings.npy", embeddings)
print("embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 320/320 [00:58<00:00,  5.51it/s]

embeddings shape: (20452, 384)


## 3. Fit BERTopic

Settings and why:
- **UMAP** compresses 384 dimensions to 5 so clustering works better. `random_state=42` makes results reproducible.
- **HDBSCAN** finds clusters of varying size and marks posts that fit no topic as outliers (`-1`). `min_cluster_size` is the smallest allowed topic. Larger means fewer, broader topics.
- **CountVectorizer** with English stop words stops words like "the" and "is" from dominating topic descriptions. `ngram_range=(1, 2)` allows two-word phrases such as "open source".

In [64]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

MIN_TOPIC_SIZE = 30   # tune this: raise for fewer, broader topics

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)

vectorizer_model = CountVectorizer(stop_words="english", min_df=2,
                                   ngram_range=(1, 2))

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs, embeddings)
posts["topic"] = topics

print("number of topics (excluding outliers):", len(set(topics)) - (1 if -1 in topics else 0))
print("outlier share:", round((np.array(topics) == -1).mean() * 100, 1), "%")

2026-11-23 01:42:19,155 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-23 01:42:30,184 - BERTopic - Dimensionality - Completed ✓
2026-11-23 01:42:30,185 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-23 01:42:33,149 - BERTopic - Cluster - Completed ✓
2026-11-23 01:42:33,183 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-23 01:42:33,746 - BERTopic - Representation - Completed ✓


number of topics (excluding outliers): 69
outlier share: 31.5 %


## 4. Look at the topics

The most important check is human: do the top words of each topic describe something coherent? Numbers come later.

In [65]:
topic_info = topic_model.get_topic_info()
topic_info[["Topic", "Count", "Name"]].head(20)

,Topic,Count,Name
0,-1,6438,-1_ai_like_people_research
1,0,954,0_scholarship_students_teaching_university
2,1,712,1_netflix_movie_watch_watching
3,2,613,2_nasa_space_spacex_moon
4,3,508,3_vaccine_measles_vaccines_covid
5,4,499,4_mental_mental health_health_people
6,5,488,5_music_song_nowplaying_youtu
7,6,478,6_healthcare_universal_universal healthcare_trump
8,7,460,7_olympics_sports_olympic_women
9,8,455,8_stock_stock market_market_stocks


In [66]:
topic_model.get_topic(0)                       # top words with scores
topic_model.get_representative_docs(0)         # example posts

['She is! She has a lot of potential for a D1 scholarship',
 'Whose literary scholarship is least genealogical?',
 "he's gonna get a scholarship"]

## 5. Do topics match our two domains?

We collected posts using tech keywords and politics keywords. A good model should mostly separate them without being told. This is a sanity check, not the final goal, because the interesting topics are the ones that mix both.

In [67]:
domain_by_topic = pd.crosstab(posts["topic"], posts["seed_domain"])
domain_by_topic["share_tech"] = (
    domain_by_topic.get("tech", 0) /
    (domain_by_topic.get("tech", 0) + domain_by_topic.get("politics", 0))
).round(2)

domain_by_topic.drop(index=-1, errors="ignore").head(20)

seed_domain,climate,education,entertainment,finance,health,politics,science,sports,tech,share_tech
topic,,,,,,,,,,
0,8,833,8,6,8,15,58,3,15,0.50
1,4,18,640,4,10,3,19,10,4,0.57
2,16,19,9,11,3,8,509,1,37,0.82
3,8,7,3,3,465,3,15,2,2,0.40
4,2,16,15,5,400,6,42,5,8,0.57
5,21,29,360,4,14,16,25,10,9,0.36
6,7,15,3,18,355,65,7,1,7,0.10
7,4,20,6,0,9,4,6,408,3,0.43
8,13,2,0,336,1,14,9,0,80,0.85


## 6. Evaluate topic quality

Two automatic metrics:
- **Coherence (c_v):** do the top words of a topic tend to appear together in the same posts? Higher is better, and values around 0.4 to 0.7 are typical.
- **Topic diversity:** the share of unique words across all topics' top words. Close to 1 means topics are distinct, and close to 0 means topics repeat the same words.

In [68]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

analyzer = vectorizer_model.build_analyzer()
tokens = [analyzer(d) for d in docs]
dictionary = Dictionary(tokens)

topic_words = []
for t in topic_model.get_topics():
    if t == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(t)][:10]
    words = [w for w in words if w in dictionary.token2id]
    if len(words) >= 2:
        topic_words.append(words)

coherence = CoherenceModel(topics=topic_words, texts=tokens,
                           dictionary=dictionary, coherence="c_v").get_coherence()

all_words = [w for ws in topic_words for w in ws]
diversity = len(set(all_words)) / len(all_words)

print("number of topics evaluated:", len(topic_words))
print("coherence (c_v):", round(coherence, 3))
print("topic diversity:", round(diversity, 3))

number of topics evaluated: 69
coherence (c_v): 0.712
topic diversity: 0.866


## 7. Visualize the topics

In [69]:
try:
    fig1 = topic_model.visualize_barchart(top_n_topics=10, n_words=6)
    fig1.write_html("topics_barchart.html")
    fig2 = topic_model.visualize_topics()
    fig2.write_html("topics_map.html")
    print("saved topics_barchart.html and topics_map.html (open them in your browser)")
except Exception as e:
    print("Skipping visuals:", e)

saved topics_barchart.html and topics_map.html (open them in your browser)


## 8. Reduce outliers (optional)

HDBSCAN leaves many posts without a topic. For the final analysis we want every user to have a topic profile, so we reassign outliers to their closest topic using their embeddings. We keep **both** columns: `topic` (strict, with outliers) and `topic_reduced` (every post assigned).

This is a trade-off. Reassigned posts are less certain, so we state this in the report.

In [70]:
new_topics = topic_model.reduce_outliers(docs, topics,
                                         strategy="embeddings",
                                         embeddings=embeddings)
posts["topic_reduced"] = new_topics

print("outliers before:", (posts["topic"] == -1).sum())
print("outliers after:", (posts["topic_reduced"] == -1).sum())

outliers before: 6438
outliers after: 0


## 9. Save results

`posts_with_topics.csv` goes into notebook 04, where it is joined with the network communities.

In [71]:
topic_names = topic_info.set_index("Topic")["Name"].to_dict()
posts["topic_name"] = posts["topic_reduced"].map(topic_names)

posts.to_csv("data/posts_with_topics.csv", index=False)
topic_info.to_csv("data/topic_info.csv", index=False)

print("saved:", posts.shape)

saved: (20452, 17)


## Summary

- Embedded **N** posts with `all-MiniLM-L6-v2` and clustered them with BERTopic.
- Found **K** topics with coherence **C** and diversity **D** (fill in from the outputs).
- Main topics: (list 4 to 6 in your own words after reading them).
- Topics were [mostly separated / partly mixed] between tech and politics; the mixed topics are: (name them).
- **Limitations:** results depend on `MIN_TOPIC_SIZE`; outlier reassignment adds uncertainty; the embedding model was trained mainly on formal English, so slang and sarcasm may be handled poorly.